In [ ]:
# importing necessary libraries

In [319]:
import pandas as pd
import numpy as np

In [320]:
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import plotly.express as px
%matplotlib inline

sns.set_style('darkgrid')
matplotlib.rcParams['font.size'] = 14
matplotlib.rcParams['figure.figsize'] = (10,6)
matplotlib.rcParams['figure.facecolor'] = '#00000000'

In [ ]:
# loading my train and test samples

In [321]:
train_sample = pd.read_csv(r"C:\Users\NNAMDI\Desktop\train.csv")
train_sample

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439135,439135,D755,MEDIUM,Miami Grand Prix,2023,0,49,2,8.0,17,92.638,-0.076,-15.859,0.859649,0.0,0.0
439136,439136,D731,MEDIUM,Miami Grand Prix,2023,0,49,2,5.0,1,85.890,-0.083,-4.907,0.859649,0.0,0.0
439137,439137,D716,MEDIUM,Miami Grand Prix,2023,0,49,2,18.0,1,91.644,-0.182,-56.371,0.942308,0.0,0.0
439138,439138,D665,HARD,Abu Dhabi Grand Prix,2023,0,48,3,10.0,1,89.947,-0.001,-20.721,0.827586,1.0,0.0


In [322]:
test_df = pd.read_csv(r"C:\Users\NNAMDI\Desktop\test.csv")
test_df

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188160,627300,D171,MEDIUM,Australian Grand Prix,2024,1,14,1,14.0,4,83.879,-16.919,-87.767,0.179487,-2.0
188161,627301,RUS,SOFT,Pre-Season Testing,2025,0,60,3,26.0,4,95.727,7.920,-36.485,0.789474,-3.0
188162,627302,D112,MEDIUM,Hungarian Grand Prix,2022,0,28,2,21.0,7,85.058,-14.180,-0.339,0.388889,3.0
188163,627303,D349,MEDIUM,Monaco Grand Prix,2024,0,20,2,15.0,7,80.074,-19.004,-37.967,0.256410,0.0


In [ ]:
# performing some feature engineering

In [323]:
train_df["laptime_roll3"] = train_df.groupby(["Driver","Year","Race"])["LapTime (s)"].transform(lambda x: x.rolling(3, min_periods = 1).mean())

C:\Users\NNAMDI\AppData\Local\Temp\ipykernel_11512\78670490.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["laptime_roll3"] = train_df.groupby(["Driver","Year","Race"])["LapTime (s)"].transform(lambda x: x.rolling(3, min_periods = 1).mean())


In [324]:
train_df["laptime_vs_roll3"] = train_df["LapTime (s)"] - train_df["laptime_roll3"]

C:\Users\NNAMDI\AppData\Local\Temp\ipykernel_11512\1551282548.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["laptime_vs_roll3"] = train_df["LapTime (s)"] - train_df["laptime_roll3"]


In [325]:
train_df["deg_per_lap"] = train_df["Cumulative_Degradation"] - train_df.groupby(["Year","Race","Driver"])["Cumulative_Degradation"].shift(1).fillna(0)

C:\Users\NNAMDI\AppData\Local\Temp\ipykernel_11512\2190710177.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["deg_per_lap"] = train_df["Cumulative_Degradation"] - train_df.groupby(["Year","Race","Driver"])["Cumulative_Degradation"].shift(1).fillna(0)


In [326]:
train_df["deg_acceleration"] = train_df["deg_per_lap"] - train_df.groupby(["Year","Race","Driver"])["deg_per_lap"].shift(1).fillna(0)

C:\Users\NNAMDI\AppData\Local\Temp\ipykernel_11512\1317386126.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["deg_acceleration"] = train_df["deg_per_lap"] - train_df.groupby(["Year","Race","Driver"])["deg_per_lap"].shift(1).fillna(0)


In [327]:
train_df["Pace_drop_late_race"] = train_df["LapTime_Delta"] * train_df["RaceProgress"]

C:\Users\NNAMDI\AppData\Local\Temp\ipykernel_11512\3839254391.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["Pace_drop_late_race"] = train_df["LapTime_Delta"] * train_df["RaceProgress"]


In [328]:
train_df["defensive_indicator"] = train_df["Position_Change"] * (train_df["RaceProgress"] > 0.6)

C:\Users\NNAMDI\AppData\Local\Temp\ipykernel_11512\2559969199.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["defensive_indicator"] = train_df["Position_Change"] * (train_df["RaceProgress"] > 0.6)


In [329]:
train_df["tyre_stress_index"] = train_df["TyreLife"] * train_inputs["Compound"]

C:\Users\NNAMDI\AppData\Local\Temp\ipykernel_11512\1679980595.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["tyre_stress_index"] = train_df["TyreLife"] * train_inputs["Compound"]


In [330]:
submission_df = pd.read_csv(r"C:\Users\NNAMDI\Desktop\sample_submission.csv")
submission_df

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,0
...,...,...
188160,627300,0
188161,627301,0
188162,627302,0
188163,627303,0


In [331]:
train_df.nunique()

id                        346246
Driver                       887
Compound                       5
Race                          26
Year                           3
PitStop                        2
LapNumber                     78
Stint                          8
TyreLife                      78
Position                      20
LapTime (s)                34724
LapTime_Delta              50497
Cumulative_Degradation    111813
RaceProgress                1845
Position_Change               37
PitNextLap                     2
laptime_roll3             153313
laptime_vs_roll3          101768
deg_per_lap               268017
deg_acceleration          309922
Pace_drop_late_race       290030
defensive_indicator           35
tyre_stress_index            123
dtype: int64

In [332]:
train_df.describe()

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,laptime_roll3,laptime_vs_roll3,deg_per_lap,deg_acceleration,Pace_drop_late_race,defensive_indicator,tyre_stress_index
count,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000,346246.000000
mean,219419.900574,2023.127427,0.120143,23.119611,1.811723,14.108303,9.511298,91.307407,-3.642414,-27.776399,0.346609,0.085410,0.176068,91.299662,0.007745,-2.381010,-0.170174,-0.789796,0.020500,14.687032
std,126782.834816,0.768476,0.325129,16.976675,0.964171,9.786306,5.273256,19.847320,46.769063,33.347147,0.259962,3.709816,0.380879,13.689511,13.987082,42.152549,70.816264,4.614075,1.088309,17.993656
min,0.000000,2022.000000,0.000000,1.000000,1.000000,1.000000,1.000000,67.694000,-2403.895000,-274.208000,0.012821,-18.000000,0.000000,68.412000,-808.663333,-2454.921000,-4906.603000,-969.141250,-18.000000,0.000000
25%,109612.750000,2023.000000,0.000000,9.000000,1.000000,6.000000,5.000000,82.944000,-8.620000,-43.749000,0.131579,-1.000000,0.000000,83.825667,-0.955917,-19.081000,-28.833000,-1.359800,0.000000,0.000000
50%,219262.500000,2023.000000,0.000000,19.000000,2.000000,12.000000,9.000000,90.879000,-0.150000,-21.090000,0.276316,0.000000,0.000000,90.825000,0.000000,-0.894500,-1.813000,-0.057607,0.000000,8.000000
75%,329278.500000,2024.000000,0.000000,36.000000,2.000000,20.000000,14.000000,98.688000,0.094000,-7.934000,0.527778,1.000000,0.000000,98.627000,0.673333,13.967750,26.056500,0.040907,0.000000,24.000000
max,439139.000000,2024.000000,1.000000,78.000000,8.000000,77.000000,20.000000,2502.809000,2423.932000,2410.000000,1.000000,18.000000,1.000000,902.420000,1614.673333,2451.682000,2510.062000,140.416667,17.000000,156.000000


In [333]:
train_df.isnull().sum()

id                        0
Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
laptime_roll3             0
laptime_vs_roll3          0
deg_per_lap               0
deg_acceleration          0
Pace_drop_late_race       0
defensive_indicator       0
tyre_stress_index         0
dtype: int64

In [334]:
train_sample.dtypes

id                          int64
Driver                     object
Compound                   object
Race                       object
Year                        int64
PitStop                     int64
LapNumber                   int64
Stint                       int64
TyreLife                  float64
Position                    int64
LapTime (s)               float64
LapTime_Delta             float64
Cumulative_Degradation    float64
RaceProgress              float64
Position_Change           float64
PitNextLap                float64
dtype: object

In [335]:
train_sample['Year'].unique()

array([2022, 2025, 2023, 2024])

In [336]:
train_sample["Year"].value_counts().sort_index()

Year
2022     82989
2023    136147
2024    127110
2025     92894
Name: count, dtype: int64

In [ ]:
# Splitting my Train data into Training and Validation Set

In [337]:
train_df = train_sample[train_sample["Year"]. isin([2022, 2023, 2024])]
train_df

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0
5,5,D012,HARD,Saudi Arabian Grand Prix,2022,0,35,2,26.0,5,93.734,-9.878,-98.041,0.500000,-3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439135,439135,D755,MEDIUM,Miami Grand Prix,2023,0,49,2,8.0,17,92.638,-0.076,-15.859,0.859649,0.0,0.0
439136,439136,D731,MEDIUM,Miami Grand Prix,2023,0,49,2,5.0,1,85.890,-0.083,-4.907,0.859649,0.0,0.0
439137,439137,D716,MEDIUM,Miami Grand Prix,2023,0,49,2,18.0,1,91.644,-0.182,-56.371,0.942308,0.0,0.0
439138,439138,D665,HARD,Abu Dhabi Grand Prix,2023,0,48,3,10.0,1,89.947,-0.001,-20.721,0.827586,1.0,0.0


In [338]:
val_df = train_sample[train_sample["Year"] == 2025]
val_df

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
10,10,MAS,HARD,Japanese Grand Prix,2025,1,24,2,10.0,10,92.638,-4.963,-149.171,0.315789,4.0,0.0
13,13,D012,HARD,Bahrain Grand Prix,2025,0,38,3,12.0,8,97.999,0.913,5.684,0.487179,-1.0,1.0
21,21,PEA,MEDIUM,United States Grand Prix,2025,0,16,1,16.0,8,100.096,3.038,116.811,0.205128,-3.0,0.0
23,23,HUL,INTERMEDIATE,British Grand Prix,2025,0,3,1,5.0,11,136.851,21.945,-31.851,0.039474,7.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
439102,439102,D061,MEDIUM,Chinese Grand Prix,2025,0,10,1,10.0,3,98.644,2.845,124.301,0.131579,0.0,0.0
439103,439103,D399,SOFT,Japanese Grand Prix,2025,0,1,1,1.0,17,102.934,-2.850,-110.281,0.013158,-5.0,0.0
439107,439107,D049,HARD,Canadian Grand Prix,2025,0,21,2,14.0,2,77.844,-3.971,-267.262,0.276316,3.0,0.0
439110,439110,D084,MEDIUM,Japanese Grand Prix,2025,0,13,1,13.0,3,92.937,-4.275,101.517,0.166667,1.0,0.0


In [ ]:
#  Preprocessing(Scaling and Encoding) my Train and Validation Dataset for Training

In [339]:
input_cols = train_df.columns[1:-1]
input_cols

Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change'],
      dtype='object')

In [340]:
target_col = "PitNextLap"

In [341]:
train_inputs = train_df[input_cols].copy()
val_inputs = val_df[input_cols].copy()

In [342]:
train_target = train_df[target_col].copy()
val_target = val_df[target_col].copy()

In [343]:
numeric_col = train_inputs.select_dtypes(include = np.number).columns.tolist()
numeric_col

['Year',
 'PitStop',
 'LapNumber',
 'Stint',
 'TyreLife',
 'Position',
 'LapTime (s)',
 'LapTime_Delta',
 'Cumulative_Degradation',
 'RaceProgress',
 'Position_Change']

In [344]:
categorical_col = train_inputs.select_dtypes(object).columns.tolist()
categorical_col

['Driver', 'Compound', 'Race']

In [345]:
from sklearn.preprocessing import MinMaxScaler

In [346]:
scaler = MinMaxScaler()

In [347]:
scaler.fit(train_inputs[numeric_col])

MinMaxScaler()

In [348]:
train_inputs[numeric_col] = scaler.transform(train_inputs[numeric_col])
val_inputs[numeric_col] = scaler.transform(val_inputs[numeric_col])

In [349]:
train_inputs[numeric_col] 

,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,0.0,0.0,0.636364,0.142857,0.500000,0.368421,0.004434,0.496358,0.109987,0.710575,0.638889
2,0.0,0.0,0.753247,0.285714,0.276316,0.631579,0.001335,0.496363,0.064704,0.817100,0.583333
3,0.5,0.0,0.012987,0.000000,0.013158,0.315789,0.010951,0.496408,0.099427,0.064935,0.500000
4,0.0,1.0,0.324675,0.285714,0.065789,0.052632,0.016502,0.499782,0.096889,0.352814,0.583333
5,0.0,0.0,0.441558,0.142857,0.328947,0.210526,0.010694,0.495879,0.065631,0.493506,0.416667
...,...,...,...,...,...,...,...,...,...,...,...
439135,0.5,0.0,0.623377,0.142857,0.092105,0.842105,0.010243,0.497909,0.096248,0.857826,0.500000
439136,0.5,0.0,0.623377,0.142857,0.052632,0.000000,0.007472,0.497908,0.100328,0.857826,0.500000
439137,0.5,0.0,0.623377,0.142857,0.223684,0.000000,0.009835,0.497887,0.081155,0.941558,0.500000
439138,0.5,0.0,0.610390,0.285714,0.118421,0.000000,0.009138,0.497925,0.094436,0.825347,0.527778


In [350]:
from sklearn.preprocessing import LabelEncoder

In [351]:
for encoded_col in categorical_col:
    encoder = LabelEncoder()
    train_inputs[encoded_col] = encoder.fit_transform(train_inputs[encoded_col])
    val_inputs[encoded_col] = encoder.fit_transform(val_inputs[encoded_col])

In [352]:
encoded_col = ['Driver', 'Compound', 'Race']
encoded_col

['Driver', 'Compound', 'Race']

In [353]:
x_train = pd.DataFrame(train_inputs[numeric_col + encoded_col])
x_val = pd.DataFrame(val_inputs[numeric_col + encoded_col])

In [ ]:
# Importing Libraries necessary for Training and Training my Model

In [354]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

In [391]:
model = xgb.XGBClassifier(random_state = 42, n_jobs = -1, n_estimators = 154, colsample_bytree = 0.6, max_depth = 8, learning_rate = 0.09,eval_metric = 'auc')
model.fit(x_train, train_target)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.09, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=154,
              n_jobs=-1, num_parallel_tree=None, ...)

In [392]:
prediction = model.predict_proba(x_val)[:,1]

In [393]:
roc_auc_score(val_target, prediction)

np.float64(0.9018698707693056)

In [358]:
submission_df

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,0
...,...,...
188160,627300,0
188161,627301,0
188162,627302,0
188163,627303,0


In [ ]:
# Creating New Feature on my Test Set

In [361]:
test_df["laptime_roll3"] = test_df.groupby(["Driver","Year","Race"])["LapTime (s)"].transform(lambda x: x.rolling(3, min_periods = 1).mean())
test_df["laptime_vs_roll3"] = test_df["LapTime (s)"] - test_df["laptime_roll3"]
test_df["deg_per_lap"] = test_df["Cumulative_Degradation"] - test_df.groupby(["Year","Race","Driver"])["Cumulative_Degradation"].shift(1).fillna(0)
test_df["deg_acceleration"] = test_df["deg_per_lap"] - test_df.groupby(["Year","Race","Driver"])["deg_per_lap"].shift(1).fillna(0)
test_df["Pace_drop_late_race"] = test_df["LapTime_Delta"] * test_df["RaceProgress"]
test_df["defensive_indicator"] = test_df["Position_Change"] * (test_df["RaceProgress"] > 0.6)
test_df["tyre_stress_index"] = test_df["TyreLife"] * test_df["Compound"]

In [ ]:
# Preprocessing my Test Set

In [359]:
test_df[numeric_col] = scaler.transform(test_df[numeric_col])

In [360]:
for encoded_col in categorical_col:
    encoder = LabelEncoder()
    test_df[encoded_col] = encoder.fit_transform(test_df[encoded_col])

In [362]:
encoded_col = ['Driver', 'Compound', 'Race']

In [363]:
x_test = test_df[numeric_col + encoded_col]

In [ ]:
# Creating Predictions amd adding it to the Submission Dataset for submission on Kaggle

In [394]:
test_prediction = model.predict_proba(x_test)[:,1]
test_prediction

array([0.0036352 , 0.00258539, 0.00377476, ..., 0.7418791 , 0.825458  ,
       0.01766686], dtype=float32)

In [395]:
submission_df["PitNextLap"] = test_prediction

In [396]:
submission_df

,id,PitNextLap
0,439140,0.003635
1,439141,0.002585
2,439142,0.003775
3,439143,0.111328
4,439144,0.800389
...,...,...
188160,627300,0.015311
188161,627301,0.559686
188162,627302,0.741879
188163,627303,0.825458


In [397]:
submission_df.to_parquet("Model2.parquet", index = False)